In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
!python3.10 -m venv /content/venv
!/content/venv/bin/python -m pip install --upgrade pip
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "autoawq==0.2.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready!")

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,184 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,265 kB]
Hit:13 https://ppa.launchpadcontent.net/graphic

In [3]:
import subprocess

log_file = open("/content/server.log", "w")
server_process = subprocess.Popen(
    [
        "/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--port", "8000",
        "--quantization", "awq",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "hermes",
    ],
    stdout=log_file, stderr=subprocess.STDOUT,
)
print("server launching, PID:", server_process.pid)

server launching, PID: 3380


In [4]:
import time, urllib.request, urllib.error

def wait_for_health(url="http://localhost:8000/v1/models", timeout_s=300, interval_s=5):
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print("healthy after %.0fs" % (time.time() - start))
                    return True
        except (urllib.error.URLError, ConnectionError):
            pass
        time.sleep(interval_s)
    print("timed out after %ds" % timeout_s)
    return False

wait_for_health()

healthy after 50s


True

In [5]:
prompts_content = """What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
What is quantisation, in one sentence?
Explain continuous batching versus static batching, briefly.
What is a KV cache?
Name two metrics you would track for an inference service.
What does p95 latency mean?
Explain what PagedAttention solves, in two sentences.
What is the difference between throughput and latency?
List two failure modes of a naive load-shedding client.
What is a straggler in a batched inference request?
Explain why AWQ can be faster than fp16 with fused kernels.
What does GPU memory utilisation control in vLLM?
Name one trade-off of quantising a model to 4-bit.
What is a tool call in the context of an LLM API?
Explain why a distractor prompt should not trigger a tool call.
What is the purpose of a benchmark harness?"""

with open("prompts.txt", "w") as f:
    f.write(prompts_content)
print("wrote prompts.txt,", len(prompts_content.split(chr(10))), "lines")

wrote prompts.txt, 20 lines


In [6]:
bench_py_content = '''"""Benchmark harness for the serving stack (week 3 day 5 reference)."""
from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx


@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None
    latency_s: Optional[float] = None
    completion_tokens: int = 0
    error: Optional[str] = None


async def _one_request(client, base_url, model, prompt, max_tokens):
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft = None
    tokens = 0

    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(ok=False, error=f"HTTP {response.status_code}: {text}")
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens)
    except Exception as exc:
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values, pct):
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(client, base_url, model, prompts, concurrency, requests_per_level, max_tokens):
    await _one_request(client, base_url, model, prompts[0], max_tokens)

    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index):
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(*(_guarded(i) for i in range(requests_per_level)))
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)

    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


def _load_prompts(path):
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels):
    header = f"{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  {'lat_p95':>8}  {'ok':>4}  {'err':>4}"
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value):
            return f"{value:.3f}" if value is not None else "  n/a"
        print(f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  {fmt(lv.ttft_p50_s):>9}  "
              f"{fmt(lv.ttft_p95_s):>9}  {fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}")


def _write_report(out_path, run_record):
    document = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args):
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits, headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client, base_url=args.base_url, model=args.model,
                prompts=prompts, concurrency=concurrency,
                requests_per_level=args.requests_per_level, max_tokens=args.max_tokens,
            )
            levels.append(report)
            print(f"[level {concurrency}] tok/s={report.tokens_per_s} "
                  f"ttft_p95={report.ttft_p95_s} errors={report.errors}", flush=True)

    return {
        "timestamp": int(time.time()), "base_url": args.base_url, "model": args.model,
        "requests_per_level": args.requests_per_level, "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file, "levels": [vars(lv) for lv in levels],
    }, levels


def main():
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.sample.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""))
    parser.add_argument("--timeout", type=float, default=120.0)
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()
'''

with open("bench.py", "w") as f:
    f.write(bench_py_content)
print("wrote bench.py")

wrote bench.py


In [7]:
!/content/venv/bin/python bench.py \
  --base-url http://localhost:8000 \
  --model Qwen/Qwen2.5-1.5B-Instruct-AWQ \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=94.53 ttft_p95=0.079 errors=0
[level 2] tok/s=176.24 ttft_p95=0.1566 errors=0
[level 4] tok/s=302.06 ttft_p95=0.0942 errors=0
[level 8] tok/s=491.02 ttft_p95=0.1486 errors=0
[level 16] tok/s=716.39 ttft_p95=0.2301 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     94.53      0.050      0.079     1.368    20     0
   2    176.24      0.059      0.157     1.401    20     0
   4    302.06      0.066      0.094     1.642    20     0
   8    491.02      0.064      0.149     1.932    20     0
  16    716.39      0.221      0.230     2.399    20     0

wrote bench_report.json (run appended)


In [8]:
import json
levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 1.5
under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

c= 1  tok/s=   94.5  ttft_p95=0.079  lat_p95=1.368  errors=0
c= 2  tok/s=  176.2  ttft_p95=0.157  lat_p95=1.401  errors=0
c= 4  tok/s=  302.1  ttft_p95=0.094  lat_p95=1.642  errors=0
c= 8  tok/s=  491.0  ttft_p95=0.149  lat_p95=1.932  errors=0
c=16  tok/s=  716.4  ttft_p95=0.230  lat_p95=2.399  errors=0
knee: {'concurrency': 2, 'tokens_per_s': 176.24, 'ttft_p50_s': 0.0595, 'ttft_p95_s': 0.1566, 'latency_p95_s': 1.4013, 'errors': 0, 'ok': 20, 'wall_s': 10.622}


In [9]:
with open("knee.json", "w") as f:
    json.dump({"target_p95_s": TARGET_P95_S,
               "knee_concurrency": knee["concurrency"] if knee else None}, f)
print("wrote knee.json")

wrote knee.json


In [10]:
capacity_note_content = """# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): 1.5 seconds
- Knee concurrency (highest concurrency whose p95 is still under target): 2
- Tokens per second at the knee: 176.24
- Max sustainable request rate at the target p95: approximately 2 concurrent
  requests completing within 1.5s p95, giving a safe steady rate below the
  point where latency crosses target (concurrency 4 already breaches SLO at
  1.642s)

## The limiting family

Overhead-bound at this concurrency range: tokens/s keeps climbing strongly
through concurrency 16 (94.5 to 716.4, a 7.6x increase with no sign of
flattening), which rules out a hard compute or memory ceiling being reached
yet. What breaks the SLO first is p95 latency climbing steadily as more
requests queue behind each other, consistent with per-request scheduling
and queueing overhead accumulating faster than raw throughput capacity is
exhausted.

## Why the knee, not the peak

Concurrency 16 produces the highest raw throughput (716.4 tokens/s), but at
that point p95 latency (2.399s) already blows past the 1.5s SLO by 60%,
meaning real users would be waiting too long to count as being served
correctly. The knee at concurrency 2 is the number that respects the
promise actually made to users; the peak at concurrency 16 is only
achievable by silently breaking that promise, so it cannot be reported as
real serving capacity.
"""

with open("capacity-note.md", "w") as f:
    f.write(capacity_note_content.strip())
print("capacity-note.md written successfully.")

capacity-note.md written successfully.


In [11]:
import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    pass


def fail(reason):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main():
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found")
    with open("bench_report.json") as fh:
        document = json.load(fh)

    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output or a bare list")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    if not os.path.exists("knee.json"):
        fail("knee.json not found")
    with open("knee.json") as fh:
        knee = json.load(fh)
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty")

    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors and no explanation in capacity-note.md")

    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    pass

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [12]:
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md", "knee.json"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import torch
print("model in memory:", "model" in dir())

model in memory: False


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda")

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

print("model and tokenizer loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
!pip install -q "transformers==4.46.*" "accelerate==1.1.*"
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 42.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
done


In [3]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda")

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

print("model and tokenizer loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model and tokenizer loaded


In [4]:
results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt").to("cuda"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

128 2.621527910232544
512 1.1197009086608887
2048 1.332505702972412


In [5]:
results_reordered = {}
for context in [2048, 512, 128]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt").to("cuda"), max_new_tokens=32)
    results_reordered[context] = time.time() - t0
    print(context, results_reordered[context])

2048 1.38444185256958
512 1.0929679870605469
128 1.0846593379974365


In [6]:
_ = model.generate(**tok(prompt_of_len(64), return_tensors="pt").to("cuda"), max_new_tokens=8)

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt").to("cuda"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

128 1.087787389755249
512 1.0852060317993164
2048 1.5314726829528809


In [7]:
assert results[128] < results[512] < results[2048], (
    f"expected latency to climb with context length, got {results}"
)
print("GREEN CHECK: PASS")

AssertionError: expected latency to climb with context length, got {128: 1.087787389755249, 512: 1.0852060317993164, 2048: 1.5314726829528809}

In [8]:
_ = model.generate(**tok(prompt_of_len(128), return_tensors="pt").to("cuda"), max_new_tokens=8)

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt").to("cuda"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])

assert results[128] < results[512] < results[2048], (
    f"expected latency to climb with context length, got {results}"
)
print("GREEN CHECK: PASS")

128 1.0553901195526123
512 1.0880498886108398
2048 1.3592216968536377
GREEN CHECK: PASS


In [9]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 94.53, 'ttft_p50_s': 0.0498, 'ttft_p95_s': 0.079, 'latency_p95_s': 1.3678, 'errors': 0, 'ok': 20, 'wall_s': 19.802}
{'concurrency': 2, 'tokens_per_s': 176.24, 'ttft_p50_s': 0.0595, 'ttft_p95_s': 0.1566, 'latency_p95_s': 1.4013, 'errors': 0, 'ok': 20, 'wall_s': 10.622}
{'concurrency': 4, 'tokens_per_s': 302.06, 'ttft_p50_s': 0.0662, 'ttft_p95_s': 0.0942, 'latency_p95_s': 1.6423, 'errors': 0, 'ok': 20, 'wall_s': 6.197}
{'concurrency': 8, 'tokens_per_s': 491.02, 'ttft_p50_s': 0.064, 'ttft_p95_s': 0.1486, 'latency_p95_s': 1.9316, 'errors': 0, 'ok': 20, 'wall_s': 3.812}
{'concurrency': 16, 'tokens_per_s': 716.39, 'ttft_p50_s': 0.2213, 'ttft_p95_s': 0.2301, 'latency_p95_s': 2.3993, 'errors': 0, 'ok': 20, 'wall_s': 2.715}


In [10]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   94.5  p95=1.37s  $/M tok=$1.0285
c= 2  tok/s=  176.2  p95=1.40s  $/M tok=$0.5516
c= 4  tok/s=  302.1  p95=1.64s  $/M tok=$0.3219
c= 8  tok/s=  491.0  p95=1.93s  $/M tok=$0.198
c=16  tok/s=  716.4  p95=2.40s  $/M tok=$0.1357


In [11]:
TARGET_P95_S = 1.5

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 2, 'tokens_per_s': 176.24, 'ttft_p50_s': 0.0595, 'ttft_p95_s': 0.1566, 'latency_p95_s': 1.4013, 'errors': 0, 'ok': 20, 'wall_s': 10.622, 'cost_per_million_tokens_usd': 0.5516}
cheapest $/M token level past the knee (SLO-violating): {'concurrency': 16, 'tokens_per_s': 716.39, 'ttft_p50_s': 0.2213, 'ttft_p95_s': 0.2301, 'latency_p95_s': 2.3993, 'errors': 0, 'ok': 20, 'wall_s': 2.715, 'cost_per_million_tokens_usd': 0.1357}
-> cheaper on paper, but its p95 already exceeds your SLO -- not real usable capacity at your target.


In [12]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 176.24, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 1.4013}
{'required_tokens_per_s': 264.36, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.4013}
{'required_tokens_per_s': 352.48, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.4013}
{'required_tokens_per_s': 528.72, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 1.4013}


In [13]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("wrote cost_report.json")

wrote cost_report.json


In [14]:
import json, math, os


class _Stop(Exception):
    pass


def _fail(reason):
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def main():
    if not os.path.isfile("cost_report.json"):
        _fail("cost_report.json not found")
    with open("cost_report.json") as f:
        r = json.load(f)

    for key in ("gpu_hourly_usd", "target_p95_s", "levels", "knee", "scale_out_plan"):
        if key not in r:
            _fail("missing key '%s'" % key)
    rate, slo = r["gpu_hourly_usd"], r["target_p95_s"]
    if not isinstance(rate, (int, float)) or rate <= 0:
        _fail("gpu_hourly_usd must be a positive dollars-per-hour figure")
    if not isinstance(slo, (int, float)) or slo <= 0:
        _fail("target_p95_s must be a positive SLO in seconds")

    levels = r["levels"]
    if not isinstance(levels, list) or len(levels) < 3:
        _fail("levels must hold the bench sweep")
    for L in levels:
        for f_ in ("concurrency", "tokens_per_s", "latency_p95_s", "cost_per_million_tokens_usd"):
            if not isinstance(L.get(f_), (int, float)):
                _fail("level %r lacks numeric %s" % (L.get("concurrency"), f_))
        if not L["tokens_per_s"] or L["tokens_per_s"] <= 0:
            _fail("level %s reports tokens_per_s <= 0" % L.get("concurrency"))
        want_cost = round(rate / (L["tokens_per_s"] * 3600 / 1_000_000), 4)
        if abs(L["cost_per_million_tokens_usd"] - want_cost) > max(0.0002, want_cost * 0.01):
            _fail("concurrency %s: cost %.4f, the formula gives %.4f"
                  % (L["concurrency"], L["cost_per_million_tokens_usd"], want_cost))

    under = [L for L in levels if L["latency_p95_s"] <= slo]
    if not under:
        _fail("no level sits under the SLO")
    want_knee = max(under, key=lambda L: L["concurrency"])
    knee = r["knee"]
    if not isinstance(knee, dict) or knee.get("concurrency") != want_knee["concurrency"]:
        _fail("knee is concurrency %s; expected concurrency %s"
              % ((knee or {}).get("concurrency"), want_knee["concurrency"]))

    plan = r["scale_out_plan"]
    want_targets = [round(want_knee["tokens_per_s"] * m, 6) for m in (1.0, 1.5, 2.0, 3.0)]
    if not isinstance(plan, list) or len(plan) != 4:
        _fail("scale_out_plan must hold the four multiples 1.0, 1.5, 2.0, 3.0")
    for row, want_req in zip(plan, want_targets):
        req = row.get("required_tokens_per_s")
        if not isinstance(req, (int, float)) or abs(req - want_req) > max(0.5, want_req * 0.01):
            _fail("plan targets must be the knee's throughput x (1, 1.5, 2, 3)")
        want_n = math.ceil(want_req / want_knee["tokens_per_s"] - 1e-9)
        if row.get("replicas_needed") != want_n:
            _fail("required %.1f tok/s: replicas_needed=%r, ceil gives %d"
                  % (req, row.get("replicas_needed"), want_n))
        want_cost = round(want_n * rate, 2)
        if abs(row.get("total_hourly_cost_usd", 1e9) - want_cost) > 0.011:
            _fail("hourly cost mismatch")
        if abs(row.get("effective_p95_s", 1e9) - want_knee["latency_p95_s"]) > 0.011:
            _fail("effective_p95_s must stay at the knee's p95")

    print("recomputed costs, knee and scale-out plan all agree")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    pass

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS


In [15]:
from google.colab import files
files.download("cost_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>